In [1]:
import os
%pwd
os.chdir('../')

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir:Path
    train_data_path:Path
    test_data_path:Path
    model_name:str
    alpha:str
    l1_ratio:float
    target_column:str

In [3]:
from WineQuality.constants import *
from WineQuality.utils.common import read_yaml,create_directories

class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):

                 self.config=read_yaml(config_filepath)
                 self.params=read_yaml(params_filepath)
                 self.schema=read_yaml(schema_filepath)                 
                 create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self)->ModelTrainerConfig:
            config=self.config.model_trainer
            params=self.params.ElasticNet
            schema=self.schema.TARGET_COLUMN

            create_directories([config.root_dir])
            model_trainer_config=ModelTrainerConfig(
                    root_dir=config.root_dir,
                    train_data_path=config.train_data_path,
                    test_data_path=config.test_data_path,
                    model_name=config.model_name,
                    alpha=params.alpha,
                    l1_ratio=params.l1_ratio,
                    target_column=schema.name,
                    
            )                 
                    
            return model_trainer_config

In [4]:
import os 
from WineQuality import logger
from sklearn.linear_model import ElasticNet
import pandas as pd
import joblib

class ModelTrainer:
    def __init__(self,config:ModelTrainerConfig):
        self.config=config

    def train(self):
        try:
            train_data=pd.read_csv(self.config.train_data_path)
            test_data=pd.read_csv(self.config.test_data_path)

            

            train_x=train_data.drop([self.config.target_column],axis=1)
            test_x=test_data.drop([self.config.target_column],axis=1)
            train_y=train_data[[self.config.target_column]]
            test_y=test_data[[self.config.target_column]]


            lr=ElasticNet(alpha=self.config.alpha,l1_ratio=self.config.l1_ratio,random_state=42)
            lr.fit(train_x,train_y)
            joblib.dump(lr,os.path.join(self.config.root_dir,self.config.model_name))
         

        except Exception as e:
            logger.error(e)
            raise e
                
            

In [5]:
try:
    config=ConfigurationManager()
    model_trainer_config=config.get_model_trainer_config()
    model_trainer=ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    logger.error(e)
    raise e

FILE: config\config.yaml
TYPE: <class 'dict'>
CONTENT: {'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/entbappy/Branching-tutorial/raw/master/winequality.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}, 'data_validation': {'root_dir': 'artifacts/data_validation', 'unzip_data_dir': 'artifacts/data_ingestion/winequality-red.csv', 'STATUS_FILE': 'artifacts/data_validation/status.txt'}, 'data_transformation': {'root_dir': 'artifacts/data_transformation', 'data_path': 'artifacts/data_ingestion/winequality-red.csv'}, 'model_trainer': {'root_dir': 'artifacts/model_trainer', 'train_data_path': 'artifacts/data_transformation/train.csv', 'test_data_path': 'artifacts/data_transformation/test.csv', 'model_name': 'model.joblib'}, 'model_evaluation': {'root_dir': 'artifacts/model_evaluation', 'test_data_path': 'artifacts/data_transformation/test.csv', 'model_path': 'art

Traceback (most recent call last):
  File "e:\WineQuality\env\Lib\site-packages\IPython\core\interactiveshell.py", line 3748, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\Kanishka\AppData\Local\Temp\ipykernel_24528\2419501605.py", line 8, in <module>
    raise e
  File "C:\Users\Kanishka\AppData\Local\Temp\ipykernel_24528\2419501605.py", line 5, in <module>
    model_trainer.train()
  File "C:\Users\Kanishka\AppData\Local\Temp\ipykernel_24528\2199507922.py", line 31, in train
    raise e
  File "C:\Users\Kanishka\AppData\Local\Temp\ipykernel_24528\2199507922.py", line 14, in train
    test_data=pd.read_csv(self.config.test_data_path)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\WineQuality\env\Lib\site-packages\pandas\io\parsers\readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\WineQuality\env\Lib\site-packages\pandas\io\parsers\readers.py", line 